# 🔍 Exploratory Data Analysis — IEEE-CIS Fraud Detection

**Objective:** Understand the structure, distributions, and key patterns in the IEEE-CIS fraud detection dataset before building our ML pipeline.

**Dataset Source:** [Kaggle IEEE-CIS Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection/data)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 50)
print('Libraries loaded ✅')

## 1. Load the Raw Data
The IEEE-CIS dataset is split into two tables:
- **train_transaction.csv** — Transaction details (amount, product, card info)
- **train_identity.csv** — Device/browser identity info

We merge them on `TransactionID`.

In [ ]:
# Load datasets
df_txn = pd.read_csv('../data/raw/train_transaction.csv')
df_id = pd.read_csv('../data/raw/train_identity.csv')

print(f'Transaction table: {df_txn.shape[0]:,} rows × {df_txn.shape[1]} columns')
print(f'Identity table:    {df_id.shape[0]:,} rows × {df_id.shape[1]} columns')

In [ ]:
# Merge on TransactionID (left join — not every txn has identity info)
df = pd.merge(df_txn, df_id, on='TransactionID', how='left')
print(f'Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Target Variable — Class Imbalance
Fraud detection datasets are always heavily imbalanced. Let's see how severe it is here.

In [ ]:
# Class distribution
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Legitimate (0): {fraud_counts[0]:>8,} ({fraud_pct[0]:.2f}%)')
print(f'  Fraud      (1): {fraud_counts[1]:>8,} ({fraud_pct[1]:.2f}%)')
print(f'\n  Imbalance ratio: {fraud_counts[0] / fraud_counts[1]:.0f}:1')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
ax[0].bar(['Legitimate', 'Fraud'], fraud_counts.values, color=colors)
ax[0].set_title('Transaction Count by Class', fontweight='bold')
ax[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    ax[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
ax[1].pie(fraud_pct.values, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%',
          colors=colors, startangle=90, explode=(0, 0.1))
ax[1].set_title('Fraud Percentage', fontweight='bold')

plt.tight_layout()
plt.show()

### Key Takeaway
The dataset has a **severe class imbalance** (~3.5% fraud). A model that always predicts "legitimate" would get ~96.5% accuracy but catch zero fraud. This is why we use **ROC-AUC and F1 Score** instead of accuracy as our evaluation metrics.

## 3. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of transaction amount (log scale)
for label, color, name in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=80, alpha=0.6, color=color, label=name)

axes[0].set_xlabel('log(Transaction Amount)')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Amount Distribution (Log Scale)', fontweight='bold')
axes[0].legend()

# Box plot comparison
df_plot = df[['TransactionAmt', 'isFraud']].copy()
df_plot['isFraud'] = df_plot['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
sns.boxplot(data=df_plot, x='isFraud', y='TransactionAmt', palette=colors, ax=axes[1],
            showfliers=False)
axes[1].set_title('Transaction Amount by Class', fontweight='bold')
axes[1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.show()

print('\nAmount Statistics:')
print(df.groupby('isFraud')['TransactionAmt'].describe().round(2))

## 4. Product Category vs Fraud

In [ ]:
product_fraud = df.groupby('ProductCD')['isFraud'].agg(['sum', 'count'])
product_fraud['fraud_rate'] = (product_fraud['sum'] / product_fraud['count'] * 100).round(2)
product_fraud = product_fraud.sort_values('fraud_rate', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(product_fraud.index, product_fraud['fraud_rate'], color=sns.color_palette('viridis', len(product_fraud)))
ax.set_title('Fraud Rate by Product Category', fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Product Code')
for bar, val in zip(bars, product_fraud['fraud_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Card Network & Device Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Card network fraud rate
card4_fraud = df.groupby('card4')['isFraud'].mean().sort_values(ascending=False) * 100
card4_fraud.plot(kind='bar', ax=axes[0], color=sns.color_palette('coolwarm', len(card4_fraud)))
axes[0].set_title('Fraud Rate by Card Network', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

# Device type fraud rate
device_fraud = df.groupby('DeviceType')['isFraud'].mean().sort_values(ascending=False) * 100
device_fraud.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71', '#3498db'])
axes[1].set_title('Fraud Rate by Device Type', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 6. Missing Values Analysis

In [ ]:
# Percentage of missing values per column (top 30)
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_top = missing[missing > 0].head(30)

fig, ax = plt.subplots(figsize=(12, 8))
missing_top.plot(kind='barh', ax=ax, color=sns.color_palette('YlOrRd_r', len(missing_top)))
ax.set_xlabel('Missing Values (%)')
ax.set_title('Top 30 Columns by Missing Value %', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\nTotal columns with missing values: {(missing > 0).sum()} out of {len(missing)}')
print(f'Columns with >50% missing: {(missing > 50).sum()}')

## 7. Transaction Time Patterns

In [ ]:
# Convert TransactionDT to hour of day
df['hour'] = (df['TransactionDT'] // 3600) % 24

fig, ax = plt.subplots(figsize=(12, 5))
fraud_by_hour = df.groupby('hour')['isFraud'].mean() * 100
ax.fill_between(fraud_by_hour.index, fraud_by_hour.values, alpha=0.3, color='#e74c3c')
ax.plot(fraud_by_hour.index, fraud_by_hour.values, color='#e74c3c', linewidth=2, marker='o', markersize=5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate by Hour of Day', fontweight='bold')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 8. Email Domain Analysis

In [ ]:
# Top email domains by fraud rate
email_counts = df['P_emaildomain'].value_counts().head(10)
email_fraud = df[df['P_emaildomain'].isin(email_counts.index)].groupby('P_emaildomain')['isFraud'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 5))
email_fraud.plot(kind='bar', ax=ax, color=sns.color_palette('RdYlGn_r', len(email_fraud)))
ax.set_title('Fraud Rate by Top 10 Email Domains', fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 9. Summary & Next Steps

### Key Findings:
1. **Class Imbalance:** ~3.5% fraud rate — need SMOTE or scale_pos_weight in XGBoost
2. **Transaction Amount:** Fraud transactions tend to have higher amounts
3. **Product Category:** Categories C and S have the highest fraud rates
4. **Device Type:** Mobile devices show slightly higher fraud rates
5. **Time Patterns:** Fraud rate varies by hour — likely higher during off-peak hours
6. **Missing Values:** Many columns have >50% missing — XGBoost handles this natively

### Next Steps:
- Run `python -m src.preprocess` to clean data and create train/test splits
- Run `python -m src.train` to train XGBoost + Random Forest with MLflow tracking
- Run `python -m src.evaluate` to generate evaluation plots
- Run `python -m src.monitor` to generate Evidently AI drift reports